# 04 — Serving & Agent

Publish swap, masked views, queries, agent mock.

In [1]:
import pathlib
from dotenv import load_dotenv; load_dotenv(dotenv_path=pathlib.Path(".env") if pathlib.Path(".env").exists() else pathlib.Path("../.env"))
# serving counts
import os; from sqlalchemy import create_engine, text
host=os.getenv("POSTGRES_HOST","localhost"); port=os.getenv("POSTGRES_PORT","5433")
user=os.getenv("POSTGRES_STREAMLIT_READER_USER","streamlit_reader"); pw=os.getenv("POSTGRES_STREAMLIT_READER_PASSWORD")
db=os.getenv("POSTGRES_WAREHOUSE_DB","banking_dw")
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    for t in ["serving.dim_customer","serving.dim_account","serving.fct_transactions","serving.customers_masked"]:
        print(t, c.execute(text(f"SELECT count(*) FROM {t}")).scalar())


serving.dim_customer 6
serving.dim_account 9
serving.fct_transactions 20
serving.customers_masked 6


In [2]:
# dashboard query
import os; from sqlalchemy import create_engine, text
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    print(c.execute(text("SELECT COUNT(*) FROM serving.fct_transactions")).scalar())
    print(list(c.execute(text("SELECT txn_date, COUNT(*), SUM(amount) FROM serving.fct_transactions GROUP BY 1 ORDER BY 1 LIMIT 3")).fetchall()))


20
[(datetime.date(2026, 9, 15), 20, Decimal('97000.00'))]


In [3]:
# agent mock
from unittest.mock import MagicMock, patch
import pandas as pd
import pathlib, sys; sys.path.insert(0, str(pathlib.Path("../").resolve()) if pathlib.Path("../agents").exists() else ".")
from agents import graph as gmod
llm = MagicMock()
llm.invoke.side_effect = [MagicMock(content="SELECT COUNT(*) as cnt FROM serving.fct_transactions"), MagicMock(content="20 transactions.")]
with patch.object(gmod, "_get_llm", return_value=llm):
    with patch("agents.graph.exec_sql_guarded", return_value=pd.DataFrame([{"cnt":20}])):
        from agents.graph import ask
        print(ask("how many transactions?")["answer"])


20 transactions.
